# Test

In [ ]:
def amongus():
    print("AMONGUS")

In [ ]:
import uproot
import awkward as ak
import numpy as np
import scipy
import scipy.stats as sp
import matplotlib
import matplotlib.pyplot as plt
import math as math
import hist
import vector
import mplhep
print("uproot version",uproot.__version__)
print("awkward version",ak.__version__)
print("numpy version",np.__version__)
print("scipy version",scipy.__version__)
print("matplotlib version",matplotlib.__version__)
print("hist version",hist.__version__)
print("vector version",vector.__version__)
print("mplhep version",mplhep.__version__)

Ouverture des fichiers

In [ ]:
pathOnline = "../data/counters.online.csv"
pathOffline = "../data/counters.offline.csv"
fileOnline =  open(pathOnline, 'r')
fileOffline = open(pathOffline, 'r')

In [ ]:
def read_file(file):
    t = file.readlines()
    r = {}
    k = []
    for i, lt in enumerate(t):
        lt = lt.rstrip('\n')       # get rid off linebreaks
        lt = lt.split(',')         # separate values
        for j in range(len(lt)):
            if i == 0:
                k = lt
                r[lt[j]] = []
            else:
                r[k[j]].append(int(lt[j]))
    return ak.Array(r)

In [ ]:
dOn = read_file(fileOnline)    #data Online
dOff = read_file(fileOffline)  #data Offline

On ne considère ici que certains runs

In [ ]:
runs = [290293,290401, 291263]
allRuns = dOff.run # tous les runs

fonctions pour le calcul de N_MB

In [ ]:
def F_online(pRuns):
    out = {"Fi": [], "erri": [], "duri": []}
    for r in pRuns:
        cd = dOn.run == r
        erri = np.sqrt( 1/dOn.cint7l0b[cd] + 1/dOn.cmul7l0b[cd] )[0] #\sqrt{(\sqrt cint7 / cint7)^2 + (\sqrt cmul7 / cmul7)^2} pour une loi de poisson
        Fi = (dOn.cint7l0b[cd] / dOn.cmul7l0b[cd])[0]
        
        out["Fi"].append(Fi)
        out["erri"].append(erri*Fi)
        out["duri"].append(dOn["duration(s)"][cd][0])
    return out



rFon = F_online(allRuns)
plt.errorbar(range(len(rFon["Fi"])), rFon["Fi"], rFon["erri"], ecolor="blue")

In [ ]:
def F_offline(pRuns):
    out = {"Fi": [], "mui": [], "PUi": [], "erri": [], "errmui": [], "errPUi": [],  "duri": []}
    for r in pRuns:
        cd = dOff.run == r

        # calcul de mu (correction de pileup)
        fLHC = 11.245e3 # Hz
        L0b_rate = dOn.cint7l0b[cd] / dOn["duration(s)"][cd] # fréquence du trigger L0b (en 1/s)
        R_PS = dOff.cint7ps[cd] / dOff.cint7all[cd] # taux d'évenements passant la physics selection

        x = (R_PS * L0b_rate / (dOn.interacting_bunches[cd] * fLHC))[0]
        mu = -np.log(1-x)
        out["mui"].append(mu)

        errx = x*np.sqrt(
            (1/dOff.cint7ps[cd] + 1/dOff.cint7all[cd]                    # erreur sur le taux R_PS
            + 1/dOn.cint7l0b[cd] + 1/(3*pow(dOn["duration(s)"][cd],2))   # erreur sur la fréquence du trigger L0b
            + 1/dOn.interacting_bunches[cd])[0]                          # erreur sur le nombre du bunches
        )
        errmu = 1/(1-x)*errx
        out["errmui"].append(errmu)
        
        F_PU = mu/(1-np.exp(-mu))    # correction de pileup
        out["PUi"].append(F_PU)

        errF_PU = abs(errmu/mu*(1-F_PU*np.exp(-mu))*F_PU)
        out["errPUi"].append(errF_PU)
        
        Fi = (dOff.cmsl7all[cd] / dOff["cmsl7all&0mul"][cd] \
              *dOff.cint7all[cd] / dOff["cint7all&0msl"][cd] \
              *F_PU )[0]
        out["Fi"].append(Fi)

        erri = np.sqrt(
            (1/dOff.cmsl7all[cd] + 1/dOff["cmsl7all&0mul"][cd]
             + 1/dOff.cint7all[cd] + 1/dOff["cint7all&0msl"][cd]
             +pow(errmu/mu*(1-F_PU*np.exp(-mu)), 2))[0]
        )
        out["erri"].append(erri*Fi)
        out["duri"].append(dOn["duration(s)"][cd][0])
    return out

rFoff = F_offline(allRuns)
plt.errorbar(range(len(rFoff["Fi"])), rFoff["Fi"], rFoff["erri"], ecolor="blue"); plt.show()

Moyenne des facteurs calculés avec pour poids la durée des runs

In [ ]:
def avgdF(pF):
    res = 0
    err = 0
    for i in range(len(pF["Fi"])):
        res += pF["Fi"][i]*pF["duri"][i]/sum(pF["duri"])
        err += pow(pF["erri"][i]/pF["Fi"][i],2) * pow(pF["duri"][i],2) / pow(sum(pF["duri"]),2)
    
    return [res, np.sqrt(err)*res]

In [ ]:
print(avgdF(rFon))
print(avgdF(rFoff))

Conjugaison des facteurs calculés

In [ ]:
def conjF(pF1, pF2, pmix = 0.5):
    F1 = avgdF(pF1)
    F2 = avgdF(pF2)
    res = pmix*F1[0] + (1-pmix)*F2[0]
    err = res*np.sqrt(
        pow(pmix*F1[1]/F1[0], 2) + pow((1-pmix)*F2[1]/F2[0], 2)
    )
    return [res, err]

In [ ]:
print(conjF(rFon, rFoff))